In [ ]:
# SET UP 
import wandb
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
import unicodedata

device = "cpu"

TRAINING_DATA = "~/Downloads/training_data_clean.csv"


In [4]:
# split 
def split_data(df):
    students = df['id'].unique()
    train_idx, test_idx = train_test_split(
        students, test_size=0.3, random_state=22
    )

    train_df = df[df['id'].isin(train_idx)]
    test_df = df[df['id'].isin(test_idx)]

    train_groups = train_df['id'].values
    test_groups = test_df['id'].values

    x_train = train_df.drop(columns=['target', 'id'])
    y_train = train_df['target']

    x_test = test_df.drop(columns=['target', 'id'])
    y_test = test_df['target']

    return x_train, x_test, y_train, y_test, train_groups, test_groups


In [12]:
# Preprocessing pipeline
# tfidf params are optimized params from Optuna 

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer, OrdinalEncoder
from typing import Dict, List
import unicodedata


text_cols = [
    'best_tasks_free',
    'subopt_tasks_free',
    'verify_method_free',
    ]


ord_cols = [
    'acad_tasks_rating',
    'subopt_freq_rating',
    'evidence_freq_rating',
    'verify_freq_rating',
    ]

likert_categories = [
    '1 — Not at all likely',
    '2 — Unlikely',
    '3 — Neutral / Unsure',
    '4 — Likely',
    '5 — Very likely'
]

freq_categories = [
    '1 — Never',
    '2 — Rarely',
    '3 — Sometimes',
    '4 — Often',
    '5 — Very often'
]

ord_categories = [
    likert_categories,  # acad_tasks_rating
    freq_categories,    # subopt_freq_rating
    freq_categories,    # evidence_freq_rating
    freq_categories,    # verify_freq_rating
]

cat_cols = [
    'best_tasks_select',
    'subopt_tasks_select',
    ]

cat_multi_select_choices = [
    "Explaining complex concepts simply",
    "Drafting professional text (e.g., emails, résumés)",
    "Brainstorming or generating creative ideas",
    "Writing or editing essays/reports",
    "Converting content between formats (e.g., LaTeX)",
    "Writing or debugging code",
    "Data processing or analysis",
    "Math computations",
]

# custom function makecorpus just to keep consistency in pipeline
class MakeCorpus(BaseEstimator, TransformerMixin):
    # required by TansformerMixin -ignore
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        # X is DataFrame with text columns
        return X.agg(' '.join, axis=1).tolist()

def preprocess_text():
    return Pipeline([
        ('imputer', SimpleImputer(strategy="constant", fill_value="")),
        ('combiner', MakeCorpus()),
        # adding optimized params from Optuna 
        ('tfidf', TfidfVectorizer(max_features=1000, 
                                  ngram_range=(1, 3), 
                                  min_df=2, 
                                  max_df=0.9130255379198275, 
                                  sublinear_tf=True, 
                                  norm='l2'))
    ])

def preprocess_ord():

    return make_pipeline(
        SimpleImputer(strategy="most_frequent"),
        OrdinalEncoder(categories=ord_categories)

    )

def _normalize(s: str) -> str:
    # normalize unicode, collapse spaces, strip
    s = unicodedata.normalize("NFKC", str(s))
    s = " ".join(s.split())
    return s.strip()

def _split_top_level_commas(s: str) -> List[str]:
    """
    Split on commas that are NOT inside parentheses.
    Example: "Drafting ... (e.g., emails, résumés), Math computations"
    -> ["Drafting ... (e.g., emails, résumés)", "Math computations"]
    """
    parts = []
    buf = []
    depth = 0
    for ch in s:
        if ch == '(':
            depth += 1
            buf.append(ch)
        elif ch == ')':
            depth = max(0, depth - 1)
            buf.append(ch)
        elif ch == ',' and depth == 0:
            parts.append("".join(buf))
            buf = []
        else:
            buf.append(ch)
    if buf:
        parts.append("".join(buf))
    return parts

class MultiSelectBinarizerPerColumn(BaseEstimator, TransformerMixin):
    """
    One-hot encode multi-select columns using a safe split and a fixed
    set of canonical choices. Clone-safe: __init__ does not modify params.
    If the original cell is NaN -> all dummies for that column are NaN.
    """
    def __init__(self, classes: List[str]):
        self.classes = classes  # DO NOT MODIFY HERE (clone-safe)

    # internal helpers use fitted attributes
    def _parse_cell(self, x) -> List[str]:
        if pd.isna(x) or (isinstance(x, str) and x.strip() == ""):
            return []
        norm = _normalize(x)
        pieces = [_normalize(p) for p in _split_top_level_commas(norm)]
        # keep only exact matches (normalized)
        return [p for p in pieces if p in self.classes_]

    def fit(self, X, y=None):
        X = pd.DataFrame(X)

        # create normalized, fitted classes (separate from init param)
        self.classes_ = [_normalize(c) for c in self.classes]

        self.mlbs_: Dict[str, MultiLabelBinarizer] = {}
        self.col_to_outcols_: Dict[str, List[str]] = {}

        for col in X.columns:
            mlb = MultiLabelBinarizer(classes=self.classes_)
            mlb.fit([[]])  # fit on fixed classes only
            self.mlbs_[col] = mlb
            self.col_to_outcols_[col] = [f"{col}__{c}" for c in mlb.classes_]
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        blocks = []
        for col in X.columns:
            mlb = self.mlbs_[col]
            outcols = self.col_to_outcols_[col]
            is_missing = X[col].isna()
            lists = X[col].apply(self._parse_cell)
            arr = mlb.transform(lists)
            df_bin = pd.DataFrame(arr, columns=outcols, index=X.index)
            df_bin.loc[is_missing, :] = np.nan  # preserve missingness for KNN later
            blocks.append(df_bin)
        return pd.concat(blocks, axis=1)

def preprocess_cat():
    return make_pipeline(MultiSelectBinarizerPerColumn(classes=cat_multi_select_choices),
                                 SimpleImputer(strategy="constant", fill_value=0))

def create_preprocessor():
    # create pipelines for each type, applied per column within each subset
    # pipelines run in tandem
    # changed names to shorter versions for param_grid referencing
    transformers = [
        ("text", preprocess_text(), text_cols), # pass in just the names of the columns for now, df specified later in full pipeline
        ("ord", preprocess_ord(), ord_cols),
         ("cat", preprocess_cat(), cat_cols),
    ]

    return ColumnTransformer(transformers=transformers)


In [13]:
# se3parate text preprocessors per column
from sklearn.pipeline import FeatureUnion
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

class ColumnExtractor(BaseEstimator, TransformerMixin):
    """Extract a single column by name or index"""
    def __init__(self, column):
        self.column = column  # Can be name or index

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        
        # Handle both name and index
        if isinstance(self.column, int):
            col = X.iloc[:, self.column]
        else:
            col = X[self.column]
            
        return [str(x) if pd.notna(x) else '' for x in col]


def preprocess_text_separate():
    """Separate TF-IDF vectorizers for each text column"""
    return FeatureUnion([
        ('best_tasks', Pipeline([
            ('extract', ColumnExtractor(column='best_tasks_free')),  # Use names
            ('tfidf', TfidfVectorizer(
                max_features=800,
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.75,
                stop_words='english'
            ))
        ])),
        ('subopt_tasks', Pipeline([
            ('extract', ColumnExtractor(column='subopt_tasks_free')),
            ('tfidf', TfidfVectorizer(
                max_features=400,
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.75,
                stop_words='english'
            ))
        ])),
        ('verify_method', Pipeline([
            ('extract', ColumnExtractor(column='verify_method_free')),
            ('tfidf', TfidfVectorizer(
                max_features=400,
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.75,
                stop_words='english'
            ))
        ])),
    ])
  
    
def create_preprocessor_separate_questions():
    # create pipelines for each type, applied per column within each subset
    # pipelines run in tandem
    # changed names to shorter versions for param_grid referencing
    transformers = [
        ("text", preprocess_text_separate(), text_cols), # pass in just the names of the columns for now, df specified later in full pipeline
        ("ord", preprocess_ord(), ord_cols),
         ("cat", preprocess_cat(), cat_cols),
    ]

    return ColumnTransformer(transformers=transformers)


In [14]:
# take out text completely 
def create_preprocessor_no_text():
    # create pipelines for each type, applied per column within each subset
    # pipelines run in tandem
    # changed names to shorter versions for param_grid referencing
    transformers = [
        ("ord", preprocess_ord(), ord_cols),
        ("cat", preprocess_cat(), cat_cols),
    ]

    return ColumnTransformer(transformers=transformers)


In [ ]:
import scipy.sparse as sp

# Pytorch Dataset Class
class MyDataset(Dataset):
    def __init__(self, X, y):
        # X is your preprocessed features (after ColumnTransformer)
        if sp.issparse(X):
            X = X.toarray()
            
        self.features = torch.FloatTensor(X)  # Convert sparse to dense, then to tensor
        self.labels = torch.LongTensor(y)  # Convert labels to tensor
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

In [8]:
# other helper functions for training, validation, early stopping - used in both fixed and tuned training
def validate_model(model, valid_dl, loss_func):
    model.eval() # turn off dropout, batchnorm 
    
    val_loss = 0.0
    num_correct = 0 
    
    with torch.inference_mode(): # doesn't compute gradients, just gives prediction
        
        for (features, labels) in valid_dl: 
            outputs = model(features)  # Exact forward pass to get predictions 
            val_loss += loss_func(outputs, labels) * labels.size(0)
            
            _, predicted = torch.max(outputs.data, 1)
            num_correct += (predicted == labels).sum().item()
    
    return val_loss / len(valid_dl.dataset), num_correct / len(valid_dl.dataset)

class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.should_stop = False
        
    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0
            
def get_pytorch_dataloader(x_train_processed, y_train_processed, x_test_processed, y_test_processed):
    '''' Create PyTorch Datasets for training and testing data.
    '''
    dataset = MyDataset(x_train_processed, y_train_processed)
    testdataset = MyDataset(x_test_processed, y_test_processed)
    
    return dataset, testdataset

def build_dataset(dataset, batch_size, dataset_type):
    if dataset_type == 'train':
        return DataLoader(dataset, batch_size=batch_size, shuffle=True)
    elif dataset_type == 'test':
        return DataLoader(dataset, batch_size=batch_size, shuffle=False)

def build_network(input_size, hidden_units, dropout_rate):
    model = nn.Sequential(
        nn.Linear(input_size, hidden_units),     
        nn.BatchNorm1d(hidden_units),            
        nn.ReLU(),                      
        nn.Dropout(dropout_rate),      
        nn.Linear(hidden_units, 3),              
    ).to(device)
    return model

def build_optimizer(model, lr):
    return optim.Adam(model.parameters(), lr=lr)

In [ ]:
# MAIN EXECUTION OF JUST TRAINING - INITIAL TEST 
# uses fixed hyperparameters, hardcoded below 

# DATA PREP =========================== 
x_train, x_test, y_train, y_test, train_groups, test_groups = split_data(df)

preprocessor = create_preprocessor()
x_train_processed = preprocessor.fit_transform(x_train)

label_encoder = LabelEncoder()
y_train_processed = label_encoder.fit_transform(y_train)

dataset = MyDataset(x_train_processed, y_train_processed)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

# test data 
x_test_processed = preprocessor.transform(x_test)

y_test_processed = label_encoder.transform(y_test)

testdataset = MyDataset(x_test_processed, y_test_processed)
test_loader = DataLoader(testdataset, batch_size=32, shuffle=False)

# MODEL  =========================== 
input_size = x_train_processed.shape[1]  # Number of features
model = nn.Sequential(
        nn.Linear(input_size, 256),     # input_size = number of features after preprocessing (~2000)
        nn.BatchNorm1d(256),            # Normalize the 256 values
        nn.ReLU(),                      # Activation function, max(0, x)                      
        nn.Dropout(0.5),               # Regularization
        # During training: ~128 active neurons (randomly selected each batch)
        # Each neuron learns to work independently to indentify patterns without other neurons 
        nn.Linear(256, 3))

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

loss_func = nn.CrossEntropyLoss()

early_stopping = EarlyStopping(patience=3)

for epoch in range(50): # pass through dataset 50 times, 18 batches per pass 
    # train
    model.train()
    for features, labels in train_loader: # Get batch (32 samples)
        optimizer.zero_grad() # Reset gradients calculations themselves from last batch
        outputs = model(features)  # Forward pass
        loss = loss_func(outputs, labels)
        loss.backward() # Backpropagation
        optimizer.step() # Update weights based on gradients, these accumulated over batch
    
    # validate  
    val_loss, val_acc = validate_model(model, test_loader, loss_func)
    print(f"Epoch {epoch+1}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    
    # prevent overfitting 
    early_stopping(val_loss)
    if early_stopping.should_stop:
        print("Early stopping triggered")
        break

Epoch 1, Val Loss: 0.9867, Val Acc: 0.5663
Epoch 2, Val Loss: 0.8142, Val Acc: 0.6827
Epoch 3, Val Loss: 0.6891, Val Acc: 0.6988
Epoch 4, Val Loss: 0.6599, Val Acc: 0.7309
Epoch 5, Val Loss: 0.6468, Val Acc: 0.6908
Epoch 6, Val Loss: 0.6548, Val Acc: 0.7189
Epoch 7, Val Loss: 0.6751, Val Acc: 0.7068
Epoch 8, Val Loss: 0.6898, Val Acc: 0.6908
Early stopping triggered


In [ ]:
import wandb
# MAIN EXECUTION OF TRAINING WITH WANDB, FOR HYPERPARAMETER TUNING
# reuses some functions from above, have to rebuild some with hyperparam options 

def preprocess_data(df):
    ''' Fit preprocessing pipeline and transform data accordingly.
    Note: even though TF-IDF params are hardcoded, fit still neds to be done bc of the following:

    TfidfVectorizer needs to:
        Build its vocabulary (which words exist in your training corpus)
        Calculate document frequencies for IDF weights
        Create the feature mapping (word → column index)

    OrdinalEncoder needs to:
        See which categories actually appear in your training data
        Map them to ordinal values [0, 1, 2, 3, 4]

    MultiLabelBinarizer needs to:
        Build the binary encoding structure for your multi-select columns
    '''
    x_train, x_test, y_train, y_test, train_groups, test_groups = split_data(df)
    # training data 
    preprocessor = create_preprocessor()
    x_train_processed = preprocessor.fit_transform(x_train)

    label_encoder = LabelEncoder()
    y_train_processed = label_encoder.fit_transform(y_train)

    # test data 
    x_test_processed = preprocessor.transform(x_test)
    y_test_processed = label_encoder.transform(y_test)
    
    return x_train_processed, y_train_processed, x_test_processed, y_test_processed

    
def train(config=None):
    with wandb.init(config=config) as run:
        config = run.config # Pass sweep configuration with the hyperparameters to run experiment with.
        
        # DATA PREP =========================== 
        x_train_processed, y_train_processed, x_test_processed, y_test_processed = preprocess_data(df)
        dataset, testdataset = get_pytorch_dataloader(x_train_processed, y_train_processed, x_test_processed, y_test_processed)
        train_loader = build_dataset(dataset, config.batch_size, 'train')
        test_loader = build_dataset(testdataset, config.batch_size, 'test')

        # MODEL  ===========================
        input_size = x_train_processed.shape[1]  # Number of features
        model = build_network(input_size, config.hidden_units, config.dropout_rate)

        optimizer = build_optimizer(model, config.lr)
        loss_func = nn.CrossEntropyLoss()

        early_stopping = EarlyStopping(patience=3)

        for epoch in range(config.epochs): 
            # train
            model.train()
            for features, labels in train_loader: # Get batch (32 samples)
                optimizer.zero_grad() # Reset gradients calculations themselves from last batch
                outputs = model(features)  # Forward pass
                loss = loss_func(outputs, labels)
                loss.backward() # Backpropagation
                optimizer.step() # Update weights based on gradients, these accumulated over batch
            
            # validate  
            val_loss, val_acc = validate_model(model, test_loader, loss_func)
            print(f"Epoch {epoch+1}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
            wandb.log({"val_loss": val_loss, "val_acc": val_acc})
            
            # prevent overfitting 
            early_stopping(val_loss)
            if early_stopping.should_stop:
                print("Early stopping triggered")
                break

# SWEEP CONFIG 
sweep_config = {
    'method': 'grid',
    'metric': {
        'name': 'val_acc', # neural net trained w gradient descent works better with minimize val loss 
        'goal': 'maximize'   
    },
    'parameters': {
        'lr': { # learning rate, how big a step we take during grad desc 
            'values': [0.1, 0.01, 0.001, 0.0001]
        },
        'batch_size': {
            'values': [16, 32, 64, 128]
        },
        'hidden_units': {
            'values': [128, 256, 512, 1024]
        },
        'dropout_rate': {
            'values': [0.3, 0.5, 0.7, 0.9]
        },
        'epochs': {
            'values': [20] # with early stopping I haven't seen it go this high
        }
    }
}   
sweep_id = wandb.sweep(sweep_config, project='GenAI_Characterization')
wandb.agent(sweep_id, function=train) 

Create sweep with ID: 9m21k08c
Sweep URL: https://wandb.ai/caroline-lyu-university-of-toronto/GenAI_Characterization/sweeps/9m21k08c


wandb: Agent Starting Run: 867b143t with config:
wandb: 	batch_size: 16
wandb: 	dropout_rate: 0.3
wandb: 	epochs: 20
wandb: 	hidden_units: 128
wandb: 	lr: 0.1


Epoch 1, Val Loss: 0.8034, Val Acc: 0.6345
Epoch 2, Val Loss: 0.7635, Val Acc: 0.6265
Epoch 3, Val Loss: 0.8049, Val Acc: 0.6546
Epoch 4, Val Loss: 0.9031, Val Acc: 0.6707
Epoch 5, Val Loss: 0.9817, Val Acc: 0.6064
Early stopping triggered


val_acc,▄▃▆█▁
val_loss,▂▁▂▅█
val_acc,0.60643
val_loss,0.98165


wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: Agent Starting Run: cf0kbvss with config:
wandb: 	batch_size: 16
wandb: 	dropout_rate: 0.3
wandb: 	epochs: 20
wandb: 	hidden_units: 128
wandb: 	lr: 0.01


...

val_acc,▁▂▅▇▇▇▇██████████
val_loss,█▇▆▆▅▄▃▂▂▁▁▁▁▁▁▁▁
val_acc,0.71486
val_loss,0.65198


wandb: Agent Starting Run: tii3caxl with config:
wandb: 	batch_size: 128
wandb: 	dropout_rate: 0.9
wandb: 	epochs: 20
wandb: 	hidden_units: 1024
wandb: 	lr: 0.0001


Epoch 1, Val Loss: 1.0939, Val Acc: 0.4297
Epoch 2, Val Loss: 1.0852, Val Acc: 0.4819
Epoch 3, Val Loss: 1.0724, Val Acc: 0.5181
Epoch 4, Val Loss: 1.0552, Val Acc: 0.5382
Epoch 5, Val Loss: 1.0323, Val Acc: 0.5502
Epoch 6, Val Loss: 1.0034, Val Acc: 0.5703
Epoch 7, Val Loss: 0.9685, Val Acc: 0.5783
Epoch 8, Val Loss: 0.9296, Val Acc: 0.6104
Epoch 9, Val Loss: 0.8899, Val Acc: 0.6386
Epoch 10, Val Loss: 0.8540, Val Acc: 0.6546
Epoch 11, Val Loss: 0.8229, Val Acc: 0.6667
Epoch 12, Val Loss: 0.7966, Val Acc: 0.6546
Epoch 13, Val Loss: 0.7770, Val Acc: 0.6667
Epoch 14, Val Loss: 0.7641, Val Acc: 0.6747
Epoch 15, Val Loss: 0.7551, Val Acc: 0.6707
Epoch 16, Val Loss: 0.7473, Val Acc: 0.6787
Epoch 17, Val Loss: 0.7422, Val Acc: 0.6747
Epoch 18, Val Loss: 0.7385, Val Acc: 0.6827
Epoch 19, Val Loss: 0.7320, Val Acc: 0.6867
Epoch 20, Val Loss: 0.7264, Val Acc: 0.6988


val_acc,▁▂▃▄▄▅▅▆▆▇▇▇▇▇▇▇▇███
val_loss,███▇▇▆▆▅▄▃▃▂▂▂▂▁▁▁▁▁
val_acc,0.6988
val_loss,0.72639


wandb: Sweep Agent: Waiting for job.
wandb: Sweep Agent: Exiting.


In [15]:
# MAIN EXECUTION OF TRAINING WITH WANDB, FOR HYPERPARAMETER TUNING, WITH GROUPKFOLD CROSS-VALIDATION

from sklearn.model_selection import GroupKFold
def preprocess_data_gkf(df):
    ''' Fit preprocessing pipeline and transform data accordingly, along with train groups 
    '''
    x_train, x_test, y_train, y_test, train_groups, _ = split_data(df)
    # training data 
    
    # preprocessor = create_preprocessor()
    # preprocessor = create_preprocessor_separate_questions()
    preprocessor = create_preprocessor_no_text()
    
    x_train_processed = preprocessor.fit_transform(x_train)

    label_encoder = LabelEncoder()
    y_train_processed = label_encoder.fit_transform(y_train)
    
    return x_train_processed, y_train_processed, train_groups

def train_gkf(config=None):
    with wandb.init(config=config) as run:
        config = run.config
        print('test')
        # DATA PREP =========================== 
        x_train_processed, y_train_processed, train_groups = preprocess_data_gkf(df)

        # GroupKFold Cross-Validation
        gkf = GroupKFold(n_splits=5)
        fold_val_accs = []
        
        for fold, (train_idx, val_idx) in enumerate(gkf.split(x_train_processed, y_train_processed, groups=train_groups)):
            print(f"\n--- Fold {fold + 1}/5 ---")
            
            # Split data for this fold
            X_fold_train = x_train_processed[train_idx]
            y_fold_train = y_train_processed[train_idx]
            X_fold_val = x_train_processed[val_idx]
            y_fold_val = y_train_processed[val_idx]
            
            # Create datasets and loaders
            dataset, testdataset = get_pytorch_dataloader(X_fold_train, y_fold_train, X_fold_val, y_fold_val)
    
            train_loader = build_dataset(dataset, config.batch_size, 'train')
            val_loader = build_dataset(testdataset, config.batch_size, 'test')
            
            # Build fresh model for this fold
            input_size = x_train_processed.shape[1]
            model = build_network(input_size, config.hidden_units, config.dropout_rate)
            optimizer = build_optimizer(model, config.lr)
            loss_func = nn.CrossEntropyLoss()
            
            early_stopping = EarlyStopping(patience=3)
            
            # Train this fold
            for epoch in range(config.epochs):
                model.train()
                for features, labels in train_loader:
                    optimizer.zero_grad()
                    outputs = model(features)
                    loss = loss_func(outputs, labels)
                    loss.backward()
                    optimizer.step()
                
                # Validate on this fold's validation set
                val_loss, val_acc = validate_model(model, val_loader, loss_func)
                print(f"Fold {fold+1}, Epoch {epoch+1}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
                
                early_stopping(val_loss)
                if early_stopping.should_stop:
                    print(f"Fold {fold+1}: Early stopping triggered")
                    break
            
            # Store final validation accuracy for this fold
            final_val_loss, final_val_acc = validate_model(model, val_loader, loss_func)
            fold_val_accs.append(final_val_acc)
            print(f"Fold {fold+1} Final Val Acc: {final_val_acc:.4f}")
        
        # Average across all folds
        mean_val_acc = np.mean(fold_val_accs)
        std_val_acc = np.std(fold_val_accs)
        
        print(f"Mean Val Acc: {mean_val_acc:.4f} ± {std_val_acc:.4f}")
        print(f"Fold Accuracies: {[f'{acc:.4f}' for acc in fold_val_accs]}")
        
        # Log to wandb
        wandb.log({
            "mean_val_acc": mean_val_acc,
            "std_val_acc": std_val_acc,
            "fold_val_accs": fold_val_accs
        })

# SWEEP CONFIG 
sweep_config = {
    'method': 'bayes',  # can change to grid but gkf takes longer per run
    'metric': {
        'name': 'mean_val_acc',  # Now using CV mean
        'goal': 'maximize'   
    },
    'parameters': {
        'lr': {
            'values': [0.01, 0.001, 0.0001]
        },
        'batch_size': {
            'values': [16, 32, 64]
        },
        'hidden_units': {
            'values': [128, 256, 512]
        },
        'dropout_rate': {
            'values': [0.3, 0.5, 0.7]
        },
        'epochs': {
            'value': 20
        }
    }
}   
sweep_id = wandb.sweep(sweep_config, project='GenAI_Characterization_GroupKFold_notext')
wandb.agent(sweep_id, function=train_gkf)  # Limit to 60 runs

Create sweep with ID: netxk42l
Sweep URL: https://wandb.ai/caroline-lyu-university-of-toronto/GenAI_Characterization_GroupKFold_notext/sweeps/netxk42l


wandb: Agent Starting Run: j604cr1d with config:
wandb: 	batch_size: 16
wandb: 	dropout_rate: 0.7
wandb: 	epochs: 20
wandb: 	hidden_units: 128
wandb: 	lr: 0.01


test

--- Fold 1/5 ---


Traceback (most recent call last):
  File "C:\Users\carol\AppData\Local\Temp\ipykernel_21652\2769379602.py", line 42, in train_gkf
    dataset, testdataset = get_pytorch_dataloader(X_fold_train, y_fold_train, X_fold_val, y_fold_val)
  File "C:\Users\carol\AppData\Local\Temp\ipykernel_21652\1468084802.py", line 41, in get_pytorch_dataloader
    dataset = MyDataset(x_train_processed, y_train_processed)
  File "C:\Users\carol\AppData\Local\Temp\ipykernel_21652\3310049524.py", line 5, in __init__
    self.features = torch.FloatTensor(X.toarray())  # Convert sparse to dense, then to tensor
AttributeError: 'numpy.ndarray' object has no attribute 'toarray'


Traceback (most recent call last):
  File "c:\Users\carol\Desktop\Characterize_GenAIModel\.venv\lib\site-packages\wandb\agents\pyagent.py", line 297, in _run_job
    self._function()
  File "C:\Users\carol\AppData\Local\Temp\ipykernel_21652\2769379602.py", line 42, in train_gkf
    dataset, testdataset = get_pytorch_dataloader(X_fold_train, y_fold_train, X_fold_val, y_fold_val)
  File "C:\Users\carol\AppData\Local\Temp\ipykernel_21652\1468084802.py", line 41, in get_pytorch_dataloader
    dataset = MyDataset(x_train_processed, y_train_processed)
  File "C:\Users\carol\AppData\Local\Temp\ipykernel_21652\3310049524.py", line 5, in __init__
    self.features = torch.FloatTensor(X.toarray())  # Convert sparse to dense, then to tensor
AttributeError: 'numpy.ndarray' object has no attribute 'toarray'

wandb: ERROR Run j604cr1d errored: 'numpy.ndarray' object has no attribute 'toarray'
wandb: Agent Starting Run: 3v4a4fco with config:
wandb: 	batch_size: 32
wandb: 	dropout_rate: 0.7
wandb: 	ep

test

--- Fold 1/5 ---


Traceback (most recent call last):
  File "C:\Users\carol\AppData\Local\Temp\ipykernel_21652\2769379602.py", line 42, in train_gkf
    dataset, testdataset = get_pytorch_dataloader(X_fold_train, y_fold_train, X_fold_val, y_fold_val)
  File "C:\Users\carol\AppData\Local\Temp\ipykernel_21652\1468084802.py", line 41, in get_pytorch_dataloader
    dataset = MyDataset(x_train_processed, y_train_processed)
  File "C:\Users\carol\AppData\Local\Temp\ipykernel_21652\3310049524.py", line 5, in __init__
    self.features = torch.FloatTensor(X.toarray())  # Convert sparse to dense, then to tensor
AttributeError: 'numpy.ndarray' object has no attribute 'toarray'


Traceback (most recent call last):
  File "c:\Users\carol\Desktop\Characterize_GenAIModel\.venv\lib\site-packages\wandb\agents\pyagent.py", line 297, in _run_job
    self._function()
  File "C:\Users\carol\AppData\Local\Temp\ipykernel_21652\2769379602.py", line 42, in train_gkf
    dataset, testdataset = get_pytorch_dataloader(X_fold_train, y_fold_train, X_fold_val, y_fold_val)
  File "C:\Users\carol\AppData\Local\Temp\ipykernel_21652\1468084802.py", line 41, in get_pytorch_dataloader
    dataset = MyDataset(x_train_processed, y_train_processed)
  File "C:\Users\carol\AppData\Local\Temp\ipykernel_21652\3310049524.py", line 5, in __init__
    self.features = torch.FloatTensor(X.toarray())  # Convert sparse to dense, then to tensor
AttributeError: 'numpy.ndarray' object has no attribute 'toarray'

wandb: Ctrl + C detected. Stopping sweep.
